# Classic ML Project Template

- Project: ____
- Date: ____
- Dataset: ____
- Author(s): ____
- Git Commit: ____

## 1. Problem Statement & Success Metrics
- Business context
- Metric (RMSE, ROC-AUC, cost-based, v.v.)
- Constraints (latency, interpretability)

## 2. Setup
Document environment versions.

In [ ]:
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
print(sys.version)

## 3. Data Loading

In [ ]:
DATA_PATH = "./data/raw.csv"
df = pd.read_csv(DATA_PATH)
df.head()

## 4. Exploratory Data Analysis (EDA)
- Missing values
- Distribution & target analysis
- Correlations

In [ ]:
df.info()
df.describe().T

In [ ]:
sns.histplot(df['target'], kde=True)
plt.show()

## 5. Feature Engineering
- Scaling
- Encoding
- Interaction/Domain features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
numerical = ['num_col1', 'num_col2']
categorical = ['cat_col1']
preprocess = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])
X = df[numerical + categorical]
y = df['target']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y)

## 6. Modeling
### 6.1 Baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
baseline = Pipeline([('preprocess', preprocess), ('model', LogisticRegression(max_iter=1000))])
baseline.fit(X_train, y_train)
preds = baseline.predict_proba(X_valid)[:, 1]
roc_auc_score(y_valid, preds)

### 6.2 Advanced Model

In [ ]:
from lightgbm import LGBMClassifier
advanced = Pipeline([('preprocess', preprocess), ('model', LGBMClassifier(n_estimators=500, learning_rate=0.05))])
advanced.fit(X_train, y_train)
preds_adv = advanced.predict_proba(X_valid)[:, 1]
roc_auc_score(y_valid, preds_adv)

## 7. Evaluation & Error Analysis

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_valid, advanced.predict(X_valid)))
confusion_matrix(y_valid, advanced.predict(X_valid))

## 8. Feature Importance / Explainability

In [ ]:
import shap
explainer = shap.TreeExplainer(advanced.named_steps['model'])
shap_values = explainer.shap_values(preprocess.transform(X_valid))
shap.summary_plot(shap_values, preprocess.transform(X_valid), feature_names=numerical + list(preprocess.named_transformers_['cat'].get_feature_names_out()))

## 9. Deployment Notes & Next Steps
- Checklist inference parity
- Monitoring plan
- Ideas cải thiện